# Mineração de Dados — PL5 (Text Mining & NLP II)

#### Nome: Fernando Jorge Silva Pires

#### Numero: PG60253


## Setup inicial

In [1]:
import math
import string
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt

import nltk
from nltk.corpus import stopwords, wordnet as wn
from nltk.tokenize import word_tokenize

import spacy
from sklearn.manifold import TSNE

import gensim.downloader as api
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess

# Downloads do NLTK
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ferna\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ferna\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\ferna\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\ferna\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [2]:
!python -m spacy download pt_core_news_sm

nlp = spacy.load("pt_core_news_sm")


Defaulting to user installation because normal site-packages is not writeable
     ---------------------------------------- 0.0/13.0 MB ? eta -:--:--
     -------- ------------------------------- 2.9/13.0 MB 21.0 MB/s eta 0:00:01
     --------------------------- ------------ 8.9/13.0 MB 25.1 MB/s eta 0:00:01
     --------------------------------------- 13.0/13.0 MB 23.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_sm')



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Parte 1 — VSM e Similaridade de Cosseno


In [3]:
documentos = [
 "O gato preto dorme no sofá da sala",
 "O cachorro late para o gato preto",
 "Gatos e cachorros são animais domésticos",
 "A sala tem um sofá confortável",
 "Animais selvagens vivem na floresta"
]

query = "gato preto no sofá"


### Questão 1.1 — Pré-processamento

Tokenização, minúsculas, remoção de stopwords e pontuação.


In [4]:
stop_words = set(stopwords.words('portuguese'))

def preprocess_text(texto: str):
    # 1) Tokenizar (minúsculas)
    tokens = word_tokenize(texto.lower(), language='portuguese')

    # 2) Remover stopwords e pontuação
    tokens_processados = [
        t for t in tokens
        if t not in stop_words
        and t not in string.punctuation
        and any(ch.isalnum() for ch in t)
    ]
    return tokens_processados

print("Teste da função:")
print(preprocess_text("O gato preto dorme no sofá!"))
print()

documentos_processados = [preprocess_text(doc) for doc in documentos]
query_processada = preprocess_text(query)

print("Documentos processados:")
for i, doc in enumerate(documentos_processados):
    print(f"Doc {i+1}: {doc}")
print(f"Query processada: {query_processada}")


Teste da função:
['gato', 'preto', 'dorme', 'sofá']

Documentos processados:
Doc 1: ['gato', 'preto', 'dorme', 'sofá', 'sala']
Doc 2: ['cachorro', 'late', 'gato', 'preto']
Doc 3: ['gatos', 'cachorros', 'animais', 'domésticos']
Doc 4: ['sala', 'sofá', 'confortável']
Doc 5: ['animais', 'selvagens', 'vivem', 'floresta']
Query processada: ['gato', 'preto', 'sofá']


### Questão 1.2 — Cálculo manual de TF‑IDF


In [5]:
def compute_tf(tokens):
    # TF(t,d) = freq(t,d) / total_tokens(d)
    total = len(tokens)
    if total == 0:
        return {}
    freq = Counter(tokens)
    return {termo: cont / total for termo, cont in freq.items()}

def compute_idf(documentos_processados):
    # IDF(t) = log(N / df(t))
    N = len(documentos_processados)
    df = {}
    for doc in documentos_processados:
        for termo in set(doc):
            df[termo] = df.get(termo, 0) + 1
    return {termo: math.log(N / freq_doc) for termo, freq_doc in df.items()}

def compute_tfidf(doc_tokens, idf_dict):
    tf = compute_tf(doc_tokens)
    tfidf = {}
    for termo, tf_val in tf.items():
        if termo in idf_dict:
            tfidf[termo] = tf_val * idf_dict[termo]
    return tfidf

idf = compute_idf(documentos_processados)

print("IDF (termos relevantes):")
for termo in ['gato', 'preto', 'sofá', 'sala', 'animais']:
    if termo in idf:
        print(f" {termo}: {idf[termo]:.4f}")
print()

vetores_docs = [compute_tfidf(doc, idf) for doc in documentos_processados]
vetor_query = compute_tfidf(query_processada, idf)

print("Vetor TF-IDF da query:")
print(vetor_query)


IDF (termos relevantes):
 gato: 0.9163
 preto: 0.9163
 sofá: 0.9163
 sala: 0.9163
 animais: 0.9163

Vetor TF-IDF da query:
{'gato': 0.3054302439580517, 'preto': 0.3054302439580517, 'sofá': 0.3054302439580517}


### Questão 1.3 — Similaridade de cosseno


In [6]:
def cosine_similarity(v1, v2):
    # cos = (v1·v2) / (||v1||*||v2||)
    termos_comuns = set(v1.keys()) & set(v2.keys())
    dot_product = sum(v1[t] * v2[t] for t in termos_comuns)

    norm1 = math.sqrt(sum(val * val for val in v1.values()))
    norm2 = math.sqrt(sum(val * val for val in v2.values()))

    if norm1 == 0 or norm2 == 0:
        return 0.0

    return dot_product / (norm1 * norm2)

print("Similaridades Query vs Documentos:")
print("-" * 60)

similaridades = []
for i, doc_vetor in enumerate(vetores_docs):
    sim = cosine_similarity(vetor_query, doc_vetor)
    similaridades.append((i, sim))
    print(f"Doc {i+1}: {sim:.4f} - '{documentos[i]}'")

similaridades_ordenadas = sorted(similaridades, key=lambda x: x[1], reverse=True)

print("\n" + "="*60)
print("RANKING DE RELEVÂNCIA:")
print("="*60)
for idx, sim in similaridades_ordenadas:
    if sim > 0:
        print(f"{sim:.4f} - Doc {idx+1}: {documentos[idx]}")


Similaridades Query vs Documentos:
------------------------------------------------------------
Doc 1: 0.6507 - 'O gato preto dorme no sofá da sala'
Doc 2: 0.4040 - 'O cachorro late para o gato preto'
Doc 3: 0.0000 - 'Gatos e cachorros são animais domésticos'
Doc 4: 0.2560 - 'A sala tem um sofá confortável'
Doc 5: 0.0000 - 'Animais selvagens vivem na floresta'

RANKING DE RELEVÂNCIA:
0.6507 - Doc 1: O gato preto dorme no sofá da sala
0.4040 - Doc 2: O cachorro late para o gato preto
0.2560 - Doc 4: A sala tem um sofá confortável


### Questão 1.4 — Como a remoção de stopwords afetou os resultados?

- Remove ruído (palavras muito frequentes e pouco informativas, ex.: **o, no, da, para**).
- Aumenta o peso relativo dos termos informativos (**gato, preto, sofá**), deixando o ranking mais coerente com a query.
- Sem remover stopwords, a similaridade pode ficar artificialmente alta entre textos só por partilharem palavras funcionais.


### Questão 1.5 — Duas limitações do modelo

1. **Bag-of-words**: ignora a ordem das palavras e estrutura sintática (negação, relações sujeito/objeto, etc.).
2. **Semântica fraca**: não lida bem com sinonímia (*cão* vs *cachorro*) nem com polissemia (vários sentidos da mesma palavra).


## Parte 2 — WordNet: Relações Semânticas


### Questão 2.1 — Explorar synsets e relações


In [7]:
def explorar_palavra(palavra, lang='por'):
    synsets = wn.synsets(palavra, lang=lang)

    if not synsets:
        print(f"Palavra '{palavra}' não encontrada no WordNet")
        return

    print(f"\n{'='*50}")
    print(f"Explorando: {palavra}")
    print(f"Número de synsets: {len(synsets)}")

    for i, syn in enumerate(synsets[:3]):
        print(f"\n--- Synset {i+1}: {syn.name()} ---")
        print("Definição:", syn.definition())

        exs = syn.examples()
        if exs:
            print("Exemplos:", exs)

        lemmas = [l.name() for l in syn.lemmas()]
        print("Lemmas:", lemmas)

        hypers = syn.hypernyms()
        if hypers:
            print("Hiperónimos:", [h.name() for h in hypers])

        hypos = syn.hyponyms()
        if hypos:
            print("Hipónimos:", [h.name() for h in hypos[:5]], ("..." if len(hypos) > 5 else ""))

palavras_teste = ['cachorro', 'computador', 'casa', 'amor', 'correr']
for palavra in palavras_teste:
    explorar_palavra(palavra)



Explorando: cachorro
Número de synsets: 2

--- Synset 1: bitch.n.04 ---
Definição: female of any member of the dog family
Lemmas: ['bitch']
Hiperónimos: ['canine.n.02']
Hipónimos: ['brood_bitch.n.01'] 

--- Synset 2: dog.n.01 ---
Definição: a member of the genus Canis (probably descended from the common wolf) that has been domesticated by man since prehistoric times; occurs in many breeds
Exemplos: ['the dog barked all night']
Lemmas: ['dog', 'domestic_dog', 'Canis_familiaris']
Hiperónimos: ['domestic_animal.n.01', 'canine.n.02']
Hipónimos: ['pooch.n.01', 'lapdog.n.01', 'spitz.n.01', 'griffon.n.02', 'toy_dog.n.01'] ...

Explorando: computador
Número de synsets: 3

--- Synset 1: calculator.n.02 ---
Definição: a small machine that is used for mathematical calculations
Lemmas: ['calculator', 'calculating_machine']
Hiperónimos: ['machine.n.01']
Hipónimos: ['hand_calculator.n.01', 'subtracter.n.02', 'adding_machine.n.01', "napier's_bones.n.01", 'abacus.n.02'] ...

--- Synset 2: computer.n.

### Questão 2.2 — Similaridade WordNet (path_similarity)


In [8]:
def similaridade_wordnet(palavra1, palavra2, lang='por'):
    syns1 = wn.synsets(palavra1, lang=lang)
    syns2 = wn.synsets(palavra2, lang=lang)

    if not syns1 or not syns2:
        return None, None

    max_sim = 0.0
    melhor_par = None

    for s1 in syns1[:3]:
        for s2 in syns2[:3]:
            try:
                sim = s1.path_similarity(s2)
                if sim is None:
                    continue
                if sim > max_sim:
                    max_sim = sim
                    melhor_par = (s1, s2)
            except Exception:
                continue

    return max_sim, melhor_par

pares = [
 ('cachorro', 'cão'),
 ('cachorro', 'gato'),
 ('cachorro', 'carro'),
 ('casa', 'residência'),
 ('casa', 'apartamento'),
 ('amor', 'ódio')
]

print("SIMILARIDADE SEMÂNTICA (WordNet)")
print("="*50)
for p1, p2 in pares:
    sim, par = similaridade_wordnet(p1, p2)
    if sim and sim > 0:
        print(f"{p1} vs {p2}: {sim:.4f}")
    else:
        print(f"{p1} vs {p2}: não foi possível calcular")


SIMILARIDADE SEMÂNTICA (WordNet)
cachorro vs cão: 1.0000
cachorro vs gato: 0.2500
cachorro vs carro: 0.0909
casa vs residência: 0.3333
casa vs apartamento: 1.0000
amor vs ódio: 0.1250


## Parte 3 — Dependency Parsing com spaCy


### Questão 3.1 — Mostrar dependências


In [9]:
def analisar_frase(frase):
    doc = nlp(frase)

    print(f"\nFrase: '{frase}'")
    print("-" * 60)
    print(f"{'Token':<15} {'POS':<10} {'Dependência':<15} {'Governador':<15}")
    print("-" * 60)

    for token in doc:
        print(f"{token.text:<15} {token.pos_:<10} {token.dep_:<15} {token.head.text:<15}")

    return doc

frases_teste = [
 "O cachorro mordeu o homem",
 "A menina de vestido azul comeu um bolo delicioso"
]

for frase in frases_teste:
    analisar_frase(frase)



Frase: 'O cachorro mordeu o homem'
------------------------------------------------------------
Token           POS        Dependência     Governador     
------------------------------------------------------------
O               DET        det             cachorro       
cachorro        NOUN       nsubj           mordeu         
mordeu          VERB       ROOT            mordeu         
o               DET        det             homem          
homem           NOUN       obj             mordeu         

Frase: 'A menina de vestido azul comeu um bolo delicioso'
------------------------------------------------------------
Token           POS        Dependência     Governador     
------------------------------------------------------------
A               DET        det             menina         
menina          NOUN       nsubj           comeu          
de              ADP        case            vestido        
vestido         NOUN       nmod            menina         
azul        

### Questão 3.2 — Extrair SVO e modificadores


In [10]:
def extrair_relacoes(doc):
    relacoes = []
    for token in doc:
        if token.pos_ == "VERB":
            verbo = token.text
            sujeitos = [c for c in token.children if c.dep_ in ["nsubj", "nsubj:pass"]]
            objetos = [c for c in token.children if c.dep_ in ["obj", "iobj", "obl"]]

            for suj in sujeitos:
                for obj in objetos:
                    relacao = {'sujeito': suj.text, 'verbo': verbo, 'objeto': obj.text}
                    relacoes.append(relacao)
                    print(f" {suj.text} → {verbo} → {obj.text}")
    return relacoes

def extrair_modificadores(doc):
    modificadores = []
    for token in doc:
        if token.pos_ == "ADJ":
            substantivo = token.head if token.head.pos_ == "NOUN" else None
            if substantivo:
                modificador = {'substantivo': substantivo.text, 'adjetivo': token.text}
                modificadores.append(modificador)
                print(f" {substantivo.text} é {token.text}")
    return modificadores

frase = "O carro vermelho e rápido ultrapassou o caminhão lento"
doc = nlp(frase)

print("Relações Sujeito-Verbo-Objeto:")
relacoes = extrair_relacoes(doc)

print("\nModificadores:")
modificadores = extrair_modificadores(doc)


Relações Sujeito-Verbo-Objeto:
 carro → ultrapassou → caminhão

Modificadores:
 carro é vermelho
 caminhão é lento
